In [1]:
# Cell 1: Imports and Setup
import mne
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import colorama
import gc
from tqdm.auto import tqdm

# Add srcs to path to import misc
sys.path.append(os.path.abspath('../srcs'))
from misc import load_eeg_data, extract_and_map_events, EXCLUDED_SUBJECTS

# Ensure matplotlib plots display inline in the notebook
%matplotlib inline

# Set MNE logging level to 'WARNING' to reduce text output clutter
mne.set_log_level('WARNING')

print(f"MNE version: {mne.__version__}")
print(f"Excluded subjects: {EXCLUDED_SUBJECTS}")

MNE version: 1.12.1
Excluded subjects: [88, 89, 92, 100, 104, 106]


## Cell 3: Data Parsing, Validation & Event Extraction


In [ ]:
# !Downloading and Parsing the Data

# Target all 109 SUBJECT_TO_TEST available in the dataset
# Note: To test quickly, change this to range(1, 3)
SUBJECT_TO_TEST = list(range(1, 110))
# SUBJECT_TO_TEST = list(range(1, 2))


In [13]:
print(f"Subjects to test: {SUBJECT_TO_TEST}")

Subjects to test: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109]


In [14]:

#! Define the runs you want to analyze.
# Ligne de base (Baseline), yeux ouverts
run_open_eyes = [1]
# Ligne de base (Baseline), yeux fermés
run_closed_eyes = [2]
# Motor execution: Open and close fist (Left vs. Right)
run_execution_hand = [3, 7, 11]
# Motor imagery: Imagine opening and closing fist (Left vs. Right)
run_imagery_hand = [4, 8, 12]
# Motor execution: Open and close both fists vs. both feet
run_execution_both_hands_feet = [5, 9, 13]
# Motor imagery: Imagine opening and closing both fists vs. both feet
run_imagery_both_hands_feet = [6, 10, 14]

# runs the entire set of runs for the BCI Competition IV dataset, which includes baseline, motor execution, and motor imagery tasks
runs = (
    run_open_eyes
    + run_closed_eyes
    + run_execution_hand
    + run_imagery_hand
    + run_execution_both_hands_feet
    + run_imagery_both_hands_feet
)

In [15]:
runs

[1, 2, 3, 7, 11, 4, 8, 12, 5, 9, 13, 6, 10, 14]

In [16]:
len(runs)

14

In [ ]:

# !Define your local data path (this should match your .gitignore)
data_path = "./mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0"

## Exploring raw data with a single subject and run 

In [18]:
raw = load_eeg_data(subject_id=1, run_id=1, base_path=data_path)

Loading data from ./mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R01.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1005 montage applied.


In [49]:
raw.get_data().shape

(64, 20000)


### **1. What `(64, 20000)` Means**

In **MNE-Python**, calling `raw.get_data()` returns a 2D NumPy array with the dimensions structured as **`(channels, time_points)`**:

* **`64` (Channels / Electrodes):**
This represents the **64 EEG scalp electrodes** recorded simultaneously using the BCI2000 international 10–10 system. Each row in the matrix corresponds to continuous voltage readings from one specific electrode (e.g., $C3$, $Cz$, $C4$, etc.).


* **`20000` (Time Points / Samples):**
This represents the total number of continuous discrete time samples recorded in that single run.

---

### **2. Calculating the Time Duration**

You can determine the exact recording duration in seconds using the dataset's sampling frequency ($f_s = 160\text{ Hz}$):

$$\text{Duration (seconds)} = \frac{\text{Total Time Points}}{\text{Sampling Rate}} = \frac{20{,}000\text{ samples}}{160\text{ samples/second}} = 125\text{ seconds}$$

* **Result:** **125 seconds** (or **2 minutes and 5 seconds**).
* **Dataset Context:** This matches the standard experimental design of the PhysioNet dataset, where each task run (Runs 3–14) lasts approximately **2 minutes** (120 seconds), plus a brief 5-second buffer at recording start/end.



---

### **3. Array Structure Summary**

| Dimension | Size | Description |
| --- | --- | --- |
| **Axis 0 (Rows)** | `64` | Electrodes/Channels recorded simultaneously across the scalp.

 |
| **Axis 1 (Columns)** | `20,000` | Temporal snapshots measured in Volts at 160 Hz.

 |

In [50]:
raw.get_data()

array([[ 0.00000000e+00, -1.18395015e-07,  1.90128761e-06, ...,
         3.58564133e-08,  1.65565110e-08, -6.45991724e-21],
       [ 8.47032947e-21, -6.65019645e-06, -8.20347217e-06, ...,
         5.23040202e-08,  2.40028442e-08, -5.51452977e-21],
       [ 1.01643954e-20, -1.51967019e-05, -2.13919905e-05, ...,
        -8.83886785e-08, -4.81269276e-08, -1.06099509e-21],
       ...,
       [ 1.69406589e-21,  3.32436552e-05,  4.19044472e-05, ...,
        -1.01252349e-07, -5.09407302e-08,  4.88989645e-21],
       [-1.10114283e-20,  2.89528318e-05,  3.70936196e-05, ...,
        -1.69872986e-07, -8.87487108e-08,  2.59688781e-21],
       [ 0.00000000e+00,  3.08974698e-05,  4.00871878e-05, ...,
        -2.37880899e-07, -1.25440095e-07,  5.08660549e-21]],
      shape=(64, 20000))

## Parsing the data and extracting statistics

In [19]:
print(
    f"Initiating download for {len(SUBJECT_TO_TEST)} SUBJECT_TO_TEST. This may take a while..."
)

# Initialize an empty list to hold our raw data objects
all_raws = []
# Iterate over each subject and run, loading the EEG data and appending it to the list
stats_list = []
# ! iterate over all the subjects specified in subject_to_test
# ?Use tqdm to provide a progress bar for the subject loading process
#  tqdm is a library that provides a fast, extensible progress bar for loops and other iterable objects.
#  It can be used to visualize the progress of long-running tasks, making it easier to monitor their status.
for subject in tqdm(SUBJECT_TO_TEST, desc="Subjects"):
    # ! the interate over all the runs specified in runs
    for run in runs:
        try:
            # !Load the EEG data for the current subject and run
            raw_subject = load_eeg_data(subject, run, base_path=data_path)

            # ──────────────────────────────────────────────────────
            # appy filters
            # ──────────────────────────────────────────────────────

            # !Set the EEG reference to 'average' and disable projection
            # ?we use the average to provide a more balanced reference across all channels, which can help reduce noise and improve signal quality.
            # ?The projection parameter is set to False in order to work directly with the raw data without applying any additional transformations or projections.
            raw_subject.set_eeg_reference("average", projection=False)
            # !Apply a band-pass filter to the EEG data, keeping frequencies between 8 and 30 Hz
            # *The band-pass filter is applied to focus on the frequency range of interest, which is alpha (8-12 Hz) and beta (13-30 Hz) bands.
            # ?These frequency bands are often associated with motor imagery and execution tasks(when thinking about movement happens), making them relevant for our analysis.
            # *fir_design='firwin' ensures perfect stability while keeping keeping waves between 8 and 30 Hz.
            # *Skip_by_annotation='edge' is used to safely cut gaps so the sound doesn't get distorded.
            raw_subject.filter(
                8.0, 30.0, fir_design="firwin", skip_by_annotation="edge"
            )
            # !Extract the data from the raw object and compute statistics for each channel
            data = raw_subject.get_data()
            for i, ch_name in enumerate(raw_subject.ch_names):
                stats_list.append(
                    {
                        "subject": subject,
                        "run": run,
                        "channel": ch_name,
                        # means values of the EEG signal for the current channel
                        "mean": np.mean(data[i]),
                        # standard deviation values of the EEG signal for the current channel
                        "std": np.std(data[i]),
                    }
                )
            # Clean up memory by deleting the raw_subject object and forcing garbage collection
            del raw_subject
            # Force garbage collection to free up memory after processing each subject and run
            gc.collect()
        except Exception:
            continue

df_stats = pd.DataFrame(stats_list)

raw = load_eeg_data(1, 4, base_path=data_path)

raw.set_eeg_reference("average", projection=False)

raw.filter(8.0, 30.0, fir_design="firwin", skip_by_annotation="edge")

print(
    f"Processing complete. Extracted metrics for {df_stats['subject'].nunique()} subjects."
)

Initiating download for 109 SUBJECT_TO_TEST. This may take a while...


Subjects:   0%|          | 0/109 [00:00<?, ?it/s]

Loading data from ./mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R01.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1005 montage applied.
Loading data from ./mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R02.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1005 montage applied.
Loading data from ./mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R03.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1005 montage applied.
Loading data from ./mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R07.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1005 montage applied.
Loading data from ./mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R11.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, standard_1005 montage applied.
Loading data from ./mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Metadata verified: 160 Hz sampling rate, 64 channels, stand

## Undestanding data

In [48]:
df_stats

,subject,run,channel,mean,std
0,1,1,FC5,-9.104718e-09,0.000011
1,1,1,FC3,-4.809224e-09,0.000011
2,1,1,FC1,-4.705209e-09,0.000011
3,1,1,FCz,-4.081466e-09,0.000011
4,1,1,FC2,-4.108152e-09,0.000010
...,...,...,...,...,...
73083,109,12,PO8,-6.241638e-10,0.000013
73084,109,12,O1,1.610058e-09,0.000014
73085,109,12,Oz,1.657497e-10,0.000015
73086,109,12,O2,-7.633742e-10,0.000015


In [45]:
# let's take only the first subjet then divice by 64 (channel) we should obtain the 14 run tests
len(list(df_stats[df_stats['subject'] == 1]['subject'])) / 64

14.0

In [40]:
df_stats['channel'].unique()

<StringArray>
['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6',  'C5',  'C3',  'C1',  'Cz',
  'C2',  'C4',  'C6', 'CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6', 'Fp1',
 'Fpz', 'Fp2', 'AF7', 'AF3', 'AFz', 'AF4', 'AF8',  'F7',  'F5',  'F3',  'F1',
  'Fz',  'F2',  'F4',  'F6',  'F8', 'FT7', 'FT8',  'T7',  'T8',  'T9', 'T10',
 'TP7', 'TP8',  'P7',  'P5',  'P3',  'P1',  'Pz',  'P2',  'P4',  'P6',  'P8',
 'PO7', 'PO3', 'POz', 'PO4', 'PO8',  'O1',  'Oz',  'O2',  'Iz']
Length: 64, dtype: str

In [35]:
df_stats['subject'].unique()

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  39,  40,
        41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,
        54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,
        67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,
        80,  81,  82,  83,  84,  85,  86,  87,  90,  91,  93,  94,  95,
        96,  97,  98,  99, 101, 102, 103, 105, 107, 108, 109])

In [46]:
raw.info.keys()

dict_keys(['acq_pars', 'acq_stim', 'ctf_head_t', 'description', 'dev_head_t', 'dev_ctf_t', 'dig', 'experimenter', 'utc_offset', 'device_info', 'file_id', 'highpass', 'hpi_subsystem', 'kit_system_id', 'helium_info', 'line_freq', 'lowpass', 'meas_date', 'meas_id', 'proj_id', 'proj_name', 'subject_info', 'xplotter_layout', 'gantry_angle', 'bads', 'chs', 'comps', 'events', 'hpi_meas', 'hpi_results', 'projs', 'proc_history', 'custom_ref_applied', 'sfreq', 'ch_names', 'nchan'])

## 🔍 Event Annotation & Marker Corruption Deep-Dive Audit

In this section, we perform a comprehensive memory-safe (`preload=False`) audit across all **109 subjects** and **14 experimental runs**.
We extract event triggers (`T0`, `T1`, `T2`) using MNE's `events_from_annotations` to verify trial counts and label integrity:
* **Baseline Runs (R01, R02)**: Verify existence of single continuous baseline annotation (1 `T0` event).
* **Motor Imagery/Execution Runs (R03–R14)**: Verify standard trial counts (30 events per run: 15 `T0` rest, 15 task triggers split between `T1` & `T2`).


In [30]:
file_path = "./mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0"

In [31]:
sub_dir = os.path.join(file_path, "S001")

In [32]:
sub_dir

'./mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001'

In [33]:
os.listdir(sub_dir)

['S001R04.edf',
 'S001R08.edf',
 'S001R12.edf',
 'S001R01.edf',
 'S001R02.edf',
 'S001R03.edf',
 'S001R07.edf',
 'S001R11.edf',
 'S001R05.edf',
 'S001R09.edf',
 'S001R13.edf',
 'S001R06.edf',
 'S001R10.edf',
 'S001R14.edf']

In [49]:
edf_files = sorted([f for f in os.listdir(sub_dir) if f.endswith(".edf")])

In [50]:
edf_files

['S001R01.edf',
 'S001R02.edf',
 'S001R03.edf',
 'S001R04.edf',
 'S001R05.edf',
 'S001R06.edf',
 'S001R07.edf',
 'S001R08.edf',
 'S001R09.edf',
 'S001R10.edf',
 'S001R11.edf',
 'S001R12.edf',
 'S001R13.edf',
 'S001R14.edf']

In [51]:
for f in  edf_files:
    file_path = os.path.join(sub_dir, f)
    # print(f.split("R")[1])
    print(f.split("R")[1].split(".")[0])
    # run_num = int(f.split("R")[1].split(".")[0])
    # print(run_num)

01
02
03
04
05
06
07
08
09
10
11
12
13
14


In [52]:

event_id = {"T0": 1, "T1": 2, "T2": 3}

In [57]:
for f in edf_files:

    file_path = os.path.join(sub_dir, f)

    raw_header = mne.io.read_raw_edf(file_path, preload=False, verbose="ERROR")

    events, _ = mne.events_from_annotations(raw_header, event_id=event_id,  verbose="ERROR")

    print(f"File: {f}, Number of Events: {len(events)}")
    print(f"time addition:{sum(events[:, 0])  }")

File: S001R01.edf, Number of Events: 1
time addition:0
File: S001R02.edf, Number of Events: 1
time addition:0
File: S001R03.edf, Number of Events: 30
time addition:288960
File: S001R04.edf, Number of Events: 30
time addition:288960
File: S001R05.edf, Number of Events: 30
time addition:288960
File: S001R06.edf, Number of Events: 30
time addition:288960
File: S001R07.edf, Number of Events: 30
time addition:288960
File: S001R08.edf, Number of Events: 30
time addition:288960
File: S001R09.edf, Number of Events: 30
time addition:288960
File: S001R10.edf, Number of Events: 30
time addition:288960
File: S001R11.edf, Number of Events: 30
time addition:288960
File: S001R12.edf, Number of Events: 30
time addition:288960
File: S001R13.edf, Number of Events: 30
time addition:288960
File: S001R14.edf, Number of Events: 30
time addition:288960


## Step 1: Centralized metadata parsing

* This cell implements the optimized extraction logic. By setting `preload=False`, we avoid loading the entire dataset into memory, which is crucial for handling large EEG datasets efficiently. The event extraction is performed using MNE's built-in functions, and we map the annotations to a custom event ID dictionary for consistency across subjects and runs. **This approach ensures that we can quickly audit and analyze the event structure without unnecessary memory overhead.**

In [2]:
# Constants
BASE_DATA_PATH = "mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0"
SUBJECTS = range(1, 110)
RUNS = range(1, 15)
EVENT_ID = {"T0": 0, "T1": 1, "T2": 2}

metadata_records = []

print(f"Starting lightweight metadata extraction for {len(SUBJECTS)} subjects...")

for sub_id in tqdm(SUBJECTS, desc="Subjects"):
    sub_str = f"S{sub_id:03d}"
    sub_dir = os.path.join(BASE_DATA_PATH, sub_str)
    
    if not os.path.exists(sub_dir):
        continue
        
    for run_id in RUNS:
        run_str = f"R{run_id:02d}"
        file_name = f"{sub_str}{run_str}.edf"
        file_path = os.path.join(sub_dir, file_name)
        
        if not os.path.exists(file_path):
            continue
            
        try:
            # lightweight parse: preload=False only reads headers/annotations
            raw_header = mne.io.read_raw_edf(file_path, preload=False, verbose="ERROR")
            
            # Extract basic metadata
            sfreq = raw_header.info["sfreq"]
            n_channels = raw_header.info["nchan"]
            
            # Extract events
            events, _ = mne.events_from_annotations(raw_header, event_id=EVENT_ID, verbose="ERROR")
            
            metadata_records.append({
                "subject": sub_id,
                "run": run_id,
                "sfreq": sfreq,
                "n_channels": n_channels,
                "events": events,
                "n_events": len(events)
            })
            
        except Exception as e:
            print(f"Error parsing {sub_str}{run_str}: {e}")

# Consolidate into a single source of truth
df_metadata = pd.DataFrame(metadata_records)
print(f"Extraction complete. Metadata captured for {len(df_metadata)} files.")
df_metadata.head()

Starting lightweight metadata extraction for 109 subjects...


Subjects:   0%|          | 0/109 [00:00<?, ?it/s]

Extraction complete. Metadata captured for 1251 files.


,subject,run,sfreq,n_channels,events,n_events
0,1,1,160.0,64,"[[0, 0, 0]]",1
1,1,2,160.0,64,"[[0, 0, 0]]",1
2,1,3,160.0,64,"[[0, 0, 0], [672, 0, 2], [1328, 0, 0], [2000, ...",30
3,1,4,160.0,64,"[[0, 0, 0], [672, 0, 2], [1328, 0, 0], [2000, ...",30
4,1,5,160.0,64,"[[0, 0, 0], [672, 0, 2], [1328, 0, 0], [2000, ...",30


## Step 2: Channel Count & Sampling Rate Validation

This step consumes the centralized `df_metadata` to identify hardware or acquisition anomalies. We enforce two strict constraints:
1. **Sampling Frequency ($f_s$):** Must be exactly **160 Hz**.
2. **Channel Count:** Must be exactly **64 channels**.

Any deviation (like Subject 88's 128 Hz sampling) will cause tensor shape mismatches and spectral distortion in the downstream pipeline.

In [3]:
# Identify subjects with non-compliant sampling rates or channel counts
df_hardware_anomalies = df_metadata[
    (df_metadata["sfreq"] != 160.0) | (df_metadata["n_channels"] != 64)
].copy()

# Group by subject to list specific reasons for exclusion
hardware_exclusion_summary = (
    df_hardware_anomalies.groupby("subject")
    .agg(
        {
            "sfreq": lambda x: (
                f"Mismatched ({x.unique()[0]} Hz)"
                if x.unique()[0] != 160.0
                else "Correct"
            ),
            "n_channels": lambda x: (
                f"Mismatched ({x.unique()[0]} Ch)" if x.unique()[0] != 64 else "Correct"
            ),
        }
    )
    .reset_index()
)

# Generate the initial exclusion list from hardware failures
hardware_excluded_subjects = sorted(
    hardware_exclusion_summary["subject"].unique().tolist()
)

print(
    f"Detected {len(hardware_excluded_subjects)} subjects with hardware/acquisition anomalies."
)
if not hardware_exclusion_summary.empty:
    display(hardware_exclusion_summary)
else:
    print("All subjects pass hardware validation.")

Detected 3 subjects with hardware/acquisition anomalies.


,subject,sfreq,n_channels
0,88,Mismatched (128.0 Hz),Correct
1,92,Mismatched (128.0 Hz),Correct
2,100,Mismatched (128.0 Hz),Correct


## Step 3: Event Annotation & Marker Corruption Analysis

In this step, we analyze the trial structure of each run using the `events` extracted in Step 1. We enforce the following dataset standards:
- **Baseline Runs (1, 2):** Exactly **1 event** (continuous recording).
- **Task Runs (3-14):** Exactly **30 events** (15 T0 rest triggers, 15 task triggers).

Additionally, we incorporate known **literature-documented anomalies** for subjects like **S038** and **S089**, where event markers are present but physiologically unreliable or desynchronized.

In [5]:
# Define rules for event counts based on run type and literature
def check_event_integrity(row):
    sub = row["subject"]
    run = row["run"]
    n_ev = row["n_events"]

    anomalies = []

    # Standard Trial Count Checks
    if run in [1, 2]:
        if n_ev != 1:
            anomalies.append(f"R{run:02d}: Baseline count mismatch ({n_ev})")
    else:
        if n_ev != 30:
            anomalies.append(f"R{run:02d}: Task count mismatch ({n_ev}/30)")

    # Known Literature Anomalies (timing drift or labeling errors)
    literature_anomalies = {
        38: "Known annotation/timing drift",
        89: "Inconsistent labeling",
        92: "Labeling errors",
        100: "Inconsistent annotations",
        104: "Inconsistent annotations",
        106: "Inconsistent annotations",
    }

    if sub in literature_anomalies:
        # We only flag the subject once to keep the summary clean
        if run == 3:
            anomalies.append(f"Literature Flag: {literature_anomalies[sub]}")

    return "; ".join(anomalies) if anomalies else None


# Apply the integrity check to the centralized metadata
df_metadata["event_anomalies"] = df_metadata.apply(check_event_integrity, axis=1)

# Summarize subjects with any event issues
df_event_anomalies = df_metadata[df_metadata["event_anomalies"].notna()].copy()
event_exclusion_summary = (
    df_event_anomalies.groupby("subject")["event_anomalies"]
    .apply(lambda x: " | ".join(sorted(list(set(filter(None, x))))))
    .reset_index()
)

event_excluded_subjects = sorted(event_exclusion_summary["subject"].unique().tolist())

# Configure pandas to show full column content (no truncation)
pd.set_option('display.max_colwidth', None)

print(
    f"Detected {len(event_excluded_subjects)} subjects with annotation or marker anomalies."
)
if not event_exclusion_summary.empty:
    display(event_exclusion_summary)
else:
    print("All subjects pass event integrity validation.")

Detected 7 subjects with annotation or marker anomalies.


,subject,event_anomalies
0,38,Literature Flag: Known annotation/timing drift
1,88,R03: Task count mismatch (38/30) | R04: Task count mismatch (38/30) | R05: Task count mismatch (38/30) | R06: Task count mismatch (38/30) | R07: Task count mismatch (38/30) | R08: Task count mismatch (38/30) | R09: Task count mismatch (38/30) | R10: Task count mismatch (38/30) | R11: Task count mismatch (38/30) | R12: Task count mismatch (38/30) | R13: Task count mismatch (38/30) | R14: Task count mismatch (38/30)
2,89,R01: Baseline count mismatch (2) | R02: Baseline count mismatch (2) | R03: Task count mismatch (44/30); Literature Flag: Inconsistent labeling
3,92,R03: Task count mismatch (38/30); Literature Flag: Labeling errors | R04: Task count mismatch (38/30) | R05: Task count mismatch (38/30) | R06: Task count mismatch (38/30) | R07: Task count mismatch (38/30) | R08: Task count mismatch (38/30) | R09: Task count mismatch (38/30) | R10: Task count mismatch (38/30) | R11: Task count mismatch (38/30) | R12: Task count mismatch (38/30) | R13: Task count mismatch (38/30) | R14: Task count mismatch (38/30)
4,100,R03: Task count mismatch (24/30); Literature Flag: Inconsistent annotations | R04: Task count mismatch (24/30) | R05: Task count mismatch (24/30) | R06: Task count mismatch (24/30) | R07: Task count mismatch (24/30) | R08: Task count mismatch (24/30) | R09: Task count mismatch (24/30) | R10: Task count mismatch (24/30) | R11: Task count mismatch (24/30) | R12: Task count mismatch (24/30) | R13: Task count mismatch (24/30) | R14: Task count mismatch (24/30)
5,104,Literature Flag: Inconsistent annotations | R08: Task count mismatch (26/30)
6,106,Literature Flag: Inconsistent annotations | R05: Task count mismatch (9/30)


## Step 4: Clean Cohort Consolidation

In this final refinement step, we merge the subjects identified in **Step 2** (hardware mismatches) and **Step 3** (annotation/marker corruption) into a single, unified **exclusion list**. 

By subtracting this list from the original cohort, we obtain the **Clean Cohort**: a group of 102 subjects with perfect tensor uniformity (64 channels, 160 Hz) and reliable event synchronization, ready for robust machine learning training.

In [7]:
# Combine all excluded subjects from Steps 2 and 3
FINAL_EXCLUDED_SUBJECTS = sorted(list(set(hardware_excluded_subjects + event_excluded_subjects)))

# Generate the valid subject list
VALID_SUBJECTS = [s for s in SUBJECTS if s not in FINAL_EXCLUDED_SUBJECTS]

# Print final report
print("==========================================")
print("       FINAL COHORT REFINEMENT REPORT     ")
print("==========================================\n")
print(f"Total Subjects Audited   : {len(SUBJECTS)}")
print(f"Excluded Subjects (Total): {len(FINAL_EXCLUDED_SUBJECTS)}")
print(f"Final Clean Cohort Size  : {len(VALID_SUBJECTS)}")
print(f"\nFinal EXCLUDED_SUBJECTS List: {FINAL_EXCLUDED_SUBJECTS}")

# Validate against the 102 subject benchmark
if len(VALID_SUBJECTS) == 102:
    print("\n✅ VALIDATION SUCCESS: Cohort matches the 102-subject benchmark.")
else:
    print(f"\n⚠️ VALIDATION WARNING: Cohort size ({len(VALID_SUBJECTS)}) differs from expected (102).")

# Update the global registry (used in downstream notebooks)
%store VALID_SUBJECTS
%store FINAL_EXCLUDED_SUBJECTS

       FINAL COHORT REFINEMENT REPORT     

Total Subjects Audited   : 109
Excluded Subjects (Total): 7
Final Clean Cohort Size  : 102

Final EXCLUDED_SUBJECTS List: [38, 88, 89, 92, 100, 104, 106]

✅ VALIDATION SUCCESS: Cohort matches the 102-subject benchmark.
Stored 'VALID_SUBJECTS' (list)
Stored 'FINAL_EXCLUDED_SUBJECTS' (list)
